In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_teddynote import logging
logging.langsmith("RAG_0928")

LangSmith 추적을 시작합니다.
[프로젝트명]
RAG_0928


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [6]:
# 문서 로드
loader = PyMuPDFLoader("data/가상Tech_업무가이드.pdf")
docs = loader.load()
docs

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-26T11:41:50+09:00', 'source': 'data/가상Tech_업무가이드.pdf', 'file_path': 'data/가상Tech_업무가이드.pdf', 'total_pages': 9, 'format': 'PDF 1.4', 'title': '가상Tech 업무 가이드', 'author': '가상Tech People Operations', 'subject': 'RAG 실습용 가상 사내 업무 가이드', 'keywords': '', 'moddate': '2026-09-26T11:41:50+09:00', 'trapped': '', 'modDate': "D:20260926114150+09'00'", 'creationDate': "D:20260926114150+09'00'", 'page': 0}, page_content='VIRTUALTECH · INTERNAL HANDBOOK\n가상Tech\n업무 가이드\n함께 일하는 방식부터 보안, 개발, 장애 대응, 복지까지\n구성원이 매일 참고하는 실무 기준\n문서 버전\n시행일\n문서 등급\n문의\nv1.0\n2026-09-01\n사내 공개\nPeople Operations\n이 문서의 목적\n가상Tech 구성원이 반복해서 찾는 사내 정책과 표준 절차를 한 문서에 정리했습니다.\n실제 회사가 아닌 RAG 검색·질의응답 실습용 가상 문서입니다.'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-26T11:41:50+09:00', 'source': 'data/가상Tech_업무가이드.pdf', 'file_path': 'data/

In [7]:
# 문서 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)

In [ ]:
#print(len(split_documents))
#print(split_documents[1].page_content)

가상Tech 업무 가이드
사내용 · v1.0
문서 책임: People Operations
02
CHAPTER 01
회사와 업무 원칙
빠르게 만들되, 고객 신뢰와 동료의 집중 시간을 지킵니다.
1.1 회사 개요
가상Tech는 기업용 협업 자동화 SaaS인 ‘FlowPilot’을 개발·운영하는 가상 기업입니다. 본사는 서울 성수동에 있으며
제품개발, 고객성공, 영업, 경영지원 조직으로 구성됩니다. 표준 근무시간은 평일 10:00-19:00이고, 핵심 협업시간은
11:00-16:00입니다.
1.2 우리가 일하는 다섯 가지 원칙
• 고객 영향부터 판단한다: 기능보다 고객 데이터와 서비스 연속성을 우선한다.
• 기록이 회의를 이긴다: 결정, 근거, 담당자, 기한을 문서에 남긴다.
• 작게 배포하고 빠르게 검증한다: 변경 범위를 줄이고 되돌릴 수 있게 만든다.
• 문제는 사람보다 시스템에서 찾는다: 회고는 비난 없이 재발 방지에 집중한다.


In [10]:
# 임베딩
embeddings = OpenAIEmbeddings()

In [ ]:
# DB 생성, 저장
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

In [13]:
# 리트리버 생성
retriever = vectorstore.as_retriever()

In [21]:
loader_txt = TextLoader("data/질문_10개.txt", encoding="utf-8")
question_txt = loader_txt.load()
question_txt

questions_txt = [line.split(". ", 1)[1] for line in question_txt[0].page_content.splitlines()
             if line.split(".")[0].isdigit()]
questions_txt

['연차를 사용하려면 최소 며칠 전에 신청해야 하나요?',
 '서울 출장 시 숙박비는 최대 얼마까지 지원되나요?',
 '출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요?',
 '회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요?',
 '재택근무를 신청하려면 어떤 조건을 충족해야 하나요?',
 '직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 어떻게 되나요?',
 '출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 언제인가요?',
 '개인 사정으로 출근 시간이 늦어질 경우 어떤 방식으로 보고해야 하나요?',
 '업무 중 시스템 장애가 발생했을 때 가장 먼저 해야 하는 조치는 무엇인가요?',
 '제주도로 2박 3일 출장을 가는 경우 숙박비 지원 한도와 출장비 정산 기한을 각각 알려주세요.']

In [22]:
retriever.invoke(questions_txt[0])

[Document(id='9f6d3f86-5cd9-47f0-a3aa-3c97cdd67377', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-26T11:41:50+09:00', 'source': 'data/가상Tech_업무가이드.pdf', 'file_path': 'data/가상Tech_업무가이드.pdf', 'total_pages': 9, 'format': 'PDF 1.4', 'title': '가상Tech 업무 가이드', 'author': '가상Tech People Operations', 'subject': 'RAG 실습용 가상 사내 업무 가이드', 'keywords': '', 'moddate': '2026-09-26T11:41:50+09:00', 'trapped': '', 'modDate': "D:20260926114150+09'00'", 'creationDate': "D:20260926114150+09'00'", 'page': 7}, page_content='• 택시는 23:00 이후 퇴근, 대중교통 미운행, 고객 일정 등 업무상 필요가 있을 때 실비 지원합니다.\n7.3 주요 복지\n제도\n지원 내용\n신청\n성장 지원\n연 120만원 교육·도서·컨퍼런스\nLearnHub에서 사전 승인\n건강 지원\n연 60만원 운동·상담\n월별 영수증 제출\n장비 지원\n3년 주기 노트북 교체, 모니터 1대\n#help-it 티켓\n리프레시\n근속 3년마다 유급 10영업일\n3개월 전 팀과 일정 협의\n휴가 중 연락\n휴가자는 업무 메시지에 응답할 의무가 없습니다. 인수인계 문서에 대체 담당자,\n진행 상태, 긴급 연락 기준을 남기고 Slack 상태와 캘린더를 갱신합니다.'),
 Document(id='63227ad7-725e-41bd-b5dd-9d9055a8fb17', metadata={'producer': 'R

In [23]:
# 프롬프트 생성
prompt = PromptTemplate.from_template(
    """너는 질문에 응답하는 역할의 비서야. 검색된 단락을 참고해. 답변을 모르겠다면 모르겠다고 답변해. 
    
    #Context:
    {context}

    #Question:
    {question}

    #Answer:
    """
)

In [24]:
# LLM 생성
llm = ChatOpenAI(model="gpt-5.6-luna", temperature=0)

# 체인 생성
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [27]:
# 체인 실행
for q in questions_txt:
    response = chain.invoke(q)
    print(f"question: {q}\nanswer: {response}\n")

question: 연차를 사용하려면 최소 며칠 전에 신청해야 하나요?
answer: 연차는 최소 1일 전에 신청해야 합니다. 단, 3일 이상 연차를 사용할 경우 2주 전에 팀과 협의해야 합니다.

question: 서울 출장 시 숙박비는 최대 얼마까지 지원되나요?
answer: 서울을 포함한 국내 출장 숙박비는 **1박당 최대 15만 원(세금 포함)**까지 지원됩니다.

question: 출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요?
answer: 출장 비용을 포함한 법인카드 사용 내역은 **지출일로부터 5영업일 이내**에 영수증과 사용 목적을 ExpenseHub에 등록해 정산해야 합니다.

question: 회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요?
answer: 제공된 문서에는 회사 노트북 분실 시 신고 절차가 구체적으로 안내되어 있지 않습니다. 문서상 기기는 회사 관리 기기만 사용하고 디스크 암호화를 유지해야 하며, 관련 보안 사고는 보안팀에 문의해야 합니다. 정확한 신고 방법은 보안팀 또는 IT 자산 담당자에게 확인하세요.

question: 재택근무를 신청하려면 어떤 조건을 충족해야 하나요?
answer: 재택근무는 **주 2회**까지 가능하며, 매주 **월요일까지 캘린더에 ‘WFH’로 표시**해야 합니다. 주 2회 기준은 주 단위라 다음 주로 이월할 수 없습니다. 다만 고객 방문이나 건강상 사유가 있으면 팀장과 People Ops의 승인이 필요합니다.

question: 직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 어떻게 되나요?
answer: 업무 관련 교육·도서·컨퍼런스 비용은 **연간 120만 원까지** 지원됩니다. 신청은 **LearnHub에서 사전 승인**을 받아야 합니다.

question: 출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 언제인가요?
answer: 출장 중 택시비는 **23:00 이후 퇴근**, **대중교통이 운행하지 않는 경우**, 또는 **고객 일정 등 업무상 필요